# 02 - Baseline: Zero-Shot mT5 Inference
## Experiment 1 - Multilingual Health QA

**Goal:** Establish a baseline score using `mT5-small` WITHOUT any fine-tuning. This gives us a reference point to measure improvement from future experiments.

**Hypothesis:** Zero-shot mT5 will perform poorly on African languages (Akan, Luganda, Amharic, Kiswahili) since it wasn't specifically trained on health QA, but should do reasonably on English subsets.

In [4]:
import pandas as pd

DATA_DIR = '/kaggle/input/datasets/jokjohnkur/health-qa-data'

train = pd.read_csv(f'{DATA_DIR}/Train.csv')
test  = pd.read_csv(f'{DATA_DIR}/Test.csv')
sample_sub = pd.read_csv(f'{DATA_DIR}/SampleSubmission.csv')

print("Sample submission columns:", sample_sub.columns.tolist())
print()
sample_sub.head(10)

Sample submission columns: ['ID', 'TargetRLF1', 'TargetR1F1', 'TargetLLM']



,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
1,ID_TS_Aka_Gha_1C80317F,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
2,ID_TS_Aka_Gha_06671AD1,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
3,ID_TS_Aka_Gha_BDD640FB,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
4,ID_TS_Aka_Gha_46685257,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
5,ID_TS_Aka_Gha_3EB13B90,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
6,ID_TS_Aka_Gha_392456DE,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
7,ID_TS_Aka_Gha_23B8C1E9,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
8,ID_TS_Aka_Gha_BC8960A9,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"
9,ID_TS_Aka_Gha_1A3D1FA7,"Wuna dey craze, eweeh","Wuna dey craze, eweeh","Wuna dey craze, eweeh"


## Load Model & Tokenizer

We start with `google/mt5-small` - a multilingual T5 model covering 101 languages including Swahili and Amharic. No fine-tuning yet; this is pure zero-shot.

In [7]:
from transformers import AutoTokenizer, MT5ForConditionalGeneration
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = MT5ForConditionalGeneration.from_pretrained(model_name).to(device)

print("Model loaded:", model_name)
print("Number of parameters:", sum(p.numel() for p in model.parameters()))

Using device: cuda


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded: google/mt5-small
Number of parameters: 556291456


## Quick Sanity Check

Before running full inference, test the model on a single example to confirm the input/output pipeline works correctly.

In [8]:
# Pick one sample from training data
sample = train.iloc[10]
print("Language:", sample['subset'])
print("Question:", sample['input'])
print("Reference Answer:", sample['output'][:200])
print()

# Tokenize and generate
inputs = tokenizer(sample['input'], return_tensors="pt", truncation=True, max_length=128).to(device)
outputs = model.generate(**inputs, max_length=128, num_beams=4)
prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("Model Prediction:", repr(prediction))

Language: Aka_Gha
Question: Ɔkwan bɛn so na metumi ne me hokafoɔ adi nkitaho yiye wɔ nna mu apɔmuden ne asane nsoano so?
Reference Answer: Sɛ wo ne wo hokafoɔ  rekasa fa nna mu apɔmuden ne asane nsoano ho a, ɛho hia sɛ: wo hyɛ nkɔmmɔbɔ no ase wɔ baabi a ɛyɛ dwoodwoo na ɛyɛ kɔkoam. Fa kasa a emu da hɔ na enni atemmuo kyerɛ wo dadwen ne w’

Model Prediction: '<extra_id_0>?'


## Key Finding: Zero-Shot mT5 Cannot Answer Questions

The output `<extra_id_0>?` is mT5's **span-corruption sentinel token** - an artifact of its pretraining objective (predicting masked spans of text), not a QA-style output. This confirms that `mt5-small` requires fine-tuning before it can perform this task at all.

**Implication:** The zero-shot baseline ROUGE scores will likely be near zero. This establishes the absolute floor we need to improve from in Experiments 2+.

### Local Validation Setup
Since the Zindi test set has no visible labels, I will create a held-out validation split from `Train.csv` to measure ROUGE-1/ROUGE-L locally before submitting to the leaderboard.

In [9]:
!pip install rouge-score -q

from sklearn.model_selection import train_test_split

# Drop the 1 row with empty input (found during EDA)
train_clean = train[train['input'].str.strip() != ''].reset_index(drop=True)

# Stratified split by language subset
train_df, val_df = train_test_split(
    train_clean, test_size=0.1, random_state=42, stratify=train_clean['subset']
)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print()
print(val_df['subset'].value_counts())

Train size: 26832
Validation size: 2982

subset
Eng_Uga    762
Aka_Gha    446
Eng_Gha    444
Eng_Eth    392
Lug_Uga    338
Eng_Ken    208
Swa_Ken    207
Amh_Eth    185
Name: count, dtype: int64


## Experiment 1 - Zero-Shot Baseline Evaluation

We sample 30 examples per language from the validation set, generate predictions with zero-shot `mt5-small`, and compute ROUGE-1 F1 and ROUGE-L F1 - matching the competition's evaluation metrics.

In [10]:
from rouge_score import rouge_scorer
import numpy as np

# Sample 30 per language for quick baseline evaluation
val_sample = val_df.groupby('subset', group_keys=False).apply(
    lambda x: x.sample(min(30, len(x)), random_state=42)
).reset_index(drop=True)

print("Validation sample size:", len(val_sample))

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=False)

predictions = []
model.eval()

for idx, row in val_sample.iterrows():
    inputs = tokenizer(row['input'], return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=128, num_beams=4)
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions.append(pred)
    if idx < 3:
        print(f"[{row['subset']}] Pred: {repr(pred)}")

val_sample['prediction'] = predictions
print("\nDone generating predictions.")

/tmp/ipykernel_58/2335725583.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  val_sample = val_df.groupby('subset', group_keys=False).apply(


Validation sample size: 240
[Aka_Gha] Pred: '<extra_id_0>.'
[Aka_Gha] Pred: '<extra_id_0>?'
[Aka_Gha] Pred: '<extra_id_0> mu'

Done generating predictions.


#### **Computing ROUGE scores**

In [11]:
rouge1_scores = []
rougeL_scores = []

for idx, row in val_sample.iterrows():
    scores = scorer.score(row['output'], row['prediction'])
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

val_sample['rouge1'] = rouge1_scores
val_sample['rougeL'] = rougeL_scores

print(f"Overall ROUGE-1 F1: {np.mean(rouge1_scores):.4f}")
print(f"Overall ROUGE-L F1: {np.mean(rougeL_scores):.4f}")
print()
print("Per-language breakdown:")
print(val_sample.groupby('subset')[['rouge1','rougeL']].mean().round(4))

Overall ROUGE-1 F1: 0.0025
Overall ROUGE-L F1: 0.0021

Per-language breakdown:
         rouge1  rougeL
subset                 
Aka_Gha  0.0096  0.0093
Amh_Eth  0.0000  0.0000
Eng_Eth  0.0056  0.0028
Eng_Gha  0.0025  0.0025
Eng_Ken  0.0010  0.0010
Eng_Uga  0.0006  0.0006
Lug_Uga  0.0011  0.0011
Swa_Ken  0.0000  0.0000


## Experiment 1: Zero-Shot mT5-Small Baseline

**Hypothesis:** Zero-shot mT5 will perform poorly since it's not instruction-tuned for QA.

**What changed:** N/A - first experiment, no fine-tuning.

**Configuration:**
- Model: google/mt5-small (556M params)
- No fine-tuning
- Generation: num_beams=4, max_length=128

**Result (validation sample, n=240):**
- ROUGE-1 F1: 0.0025
- ROUGE-L F1: 0.0021
- Worst languages: Amharic (0.0000), Kiswahili (0.0000)
- Best language: Akan (0.0096)

**Insight:** The model outputs span-corruption sentinel tokens (`<extra_id_0>`) instead of answers, confirming mT5's pretraining objective (masked span prediction) is fundamentally different from instruction-following/QA. This establishes the absolute floor — any fine-tuning should produce a dramatic improvement. The slightly higher Akan score is likely coincidental token overlap, not meaningful generation.

## Generate Test Set Submission

Run zero-shot inference on all 2,618 test examples and format the submission file per competition requirements (same prediction text in all 3 target columns).

In [12]:
test_predictions = []

for idx, row in test.iterrows():
    inputs = tokenizer(row['input'], return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_length=128, num_beams=4)
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    test_predictions.append(pred)
    if idx % 500 == 0:
        print(f"Processed {idx}/{len(test)}")

print("Done!")

Processed 0/2618
Processed 500/2618
Processed 1000/2618
Processed 1500/2618
Processed 2000/2618
Processed 2500/2618
Done!


#### **submission file**

In [13]:
submission = pd.DataFrame({
    'ID': test['ID'],
    'TargetRLF1': test_predictions,
    'TargetR1F1': test_predictions,
    'TargetLLM': test_predictions,
})

print(submission.shape)
submission.head()

(2618, 4)


,ID,TargetRLF1,TargetR1F1,TargetLLM
0,ID_TS_Aka_Gha_A3B1799D,<extra_id_0> GBV ano ma.,<extra_id_0> GBV ano ma.,<extra_id_0> GBV ano ma.
1,ID_TS_Aka_Gha_1C80317F,<extra_id_0>?eловна <extra_id_9>,<extra_id_0>?eловна <extra_id_9>,<extra_id_0>?eловна <extra_id_9>
2,ID_TS_Aka_Gha_06671AD1,<extra_id_0> bi mu ayɛ den?,<extra_id_0> bi mu ayɛ den?,<extra_id_0> bi mu ayɛ den?
3,ID_TS_Aka_Gha_BDD640FB,<extra_id_0> ?,<extra_id_0> ?,<extra_id_0> ?
4,ID_TS_Aka_Gha_46685257,<extra_id_0> mu,<extra_id_0> mu,<extra_id_0> mu


In [14]:
submission.to_csv('submission_exp1_baseline.csv', index=False)
print("Saved!")

Saved!


**Zindi Public Leaderboard Score: 0.000676** (Submission #1, screenshot saved)